ARTI308 - Machine Learning

# Project Notebook 2: Data Quality Assessment & Preprocessing
## Heart Disease Prediction

In notebook 1 we found problems that `isna()` could not see. Real data is often:
- **Incomplete** (missing values, here hidden as zeros)
- **Noisy** (outliers or random errors)
- **Inconsistent** (impossible values, wrong formats)

In this notebook we fix what can safely be fixed **before** splitting the data, and we *demonstrate* the techniques that must wait until after the split (imputation, scaling).

### Important rule: avoid data leakage
Anything that *learns* from the data (mean, median, min, max, standard deviation) must be computed on the **training set only**.
Here we use the whole dataset only to *compare strategies and understand them*.
In the model notebooks (04 to 07), the final imputation and scaling are done **inside a Pipeline after the train/test split**.

## 1. Setup and load data

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")
pd.set_option("display.max_columns", None)

DATA_PATH = "../data/raw/heart.csv"
df = pd.read_csv(DATA_PATH)

numeric_cols = ["Age", "RestingBP", "Cholesterol", "MaxHR", "Oldpeak"]
binary_cols = ["FastingBS"]
categorical_cols = ["Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope"]
target_col = "HeartDisease"

print("Original shape:", df.shape)
df.head()

## 2. Data Quality Assessment
### 2.1 Check data types

In [ ]:
df.dtypes

Numeric columns are numeric and categorical columns are text, so no type conversion is needed here (unlike the chocolate-sales lab, where `Date` and `Amount` had to be converted).

**Observation:**

*(Write what you notice after running the cell above.)*

### 2.2 Duplicate rows
Duplicates over-represent some patients and can leak between train and test.

In [ ]:
print("Duplicates:", df.duplicated().sum())
df = df.drop_duplicates()
print("Shape after dropping duplicates:", df.shape)

**Observation:**

*(Write what you notice after running the cell above.)*

## 3. Handling Missing Values
### 3.1 Convert hidden missing values to `NaN`
Zero cholesterol and zero resting blood pressure are impossible, so they are **placeholders for "not measured"**.
We replace them with `NaN` so that pandas and scikit-learn recognise them as missing.

We use a copy (`df_clean`) so the original `df` stays untouched.

In [ ]:
df_clean = df.copy()
for col in ["RestingBP", "Cholesterol"]:
    df_clean[col] = df_clean[col].replace(0, np.nan)

df_clean[["RestingBP", "Cholesterol"]].isna().sum()

**Observation:**

*(Write what you notice after running the cell above.)*

Now `isna()` can see them. What fraction of the rows is affected?

In [ ]:
missing_pct = (df_clean.isna().mean() * 100).round(2)
missing_pct[missing_pct > 0]

**Observation:**

*(Write what you notice after running the cell above.)*

### Strategy 1: Remove records
Works when only a small part of the data is missing. Here, check how much we would lose.

In [ ]:
df_remove = df_clean.dropna()
print("With NaN:   ", df_clean.shape)
print("Without NaN:", df_remove.shape)
print("Rows lost: ", round((1 - len(df_remove) / len(df_clean)) * 100, 1), "%")

**Observation:**

*(Write what you notice after running the cell above.)*

### Strategy 2: Mean imputation
Replace missing values with the average. Sensitive to outliers.

In [ ]:
df_mean = df_clean.copy()
for col in ["RestingBP", "Cholesterol"]:
    df_mean[col] = df_mean[col].fillna(df_mean[col].mean())

df_mean[["RestingBP", "Cholesterol"]].describe().round(1)

**Observation:**

*(Write what you notice after running the cell above.)*

### Strategy 3: Median imputation
Replace missing values with the middle value. More robust for skewed data and outliers.

In [ ]:
df_median = df_clean.copy()
for col in ["RestingBP", "Cholesterol"]:
    df_median[col] = df_median[col].fillna(df_median[col].median())

df_median[["RestingBP", "Cholesterol"]].describe().round(1)

**Observation:**

*(Write what you notice after running the cell above.)*

Compare the three strategies on `Cholesterol`: how do the mean and standard deviation change?

In [ ]:
comparison = pd.DataFrame({
    "original (NaN ignored)": df_clean["Cholesterol"].agg(["mean", "median", "std"]),
    "after mean imputation":  df_mean["Cholesterol"].agg(["mean", "median", "std"]),
    "after median imputation": df_median["Cholesterol"].agg(["mean", "median", "std"]),
}).round(2)
comparison

**Observation:**

*(Write what you notice after running the cell above.)*

**Decision for the project:** write here which strategy you will use and why.

> In the model notebooks (04 to 07) the chosen imputer (e.g. `SimpleImputer(strategy="median")`) is placed inside the Pipeline, so the median is learned from the training set only.

## 4. Negative `Oldpeak` values
`Oldpeak` (ST depression) is normally >= 0. A few negative values may be data-entry errors or measurement artefacts. Inspect them before deciding.

In [ ]:
neg_oldpeak = df_clean[df_clean["Oldpeak"] < 0]
print("Negative Oldpeak rows:", len(neg_oldpeak))
neg_oldpeak

**Observation:**

*(Write what you notice after running the cell above.)*

Possible decisions: keep them (they may be valid readings), set them to `NaN`, or take the absolute value.
**Write your decision and reason here.** By default the code below keeps them unchanged.

In [ ]:
# Option A (default): keep unchanged
# Option B: treat as missing
# df_clean.loc[df_clean["Oldpeak"] < 0, "Oldpeak"] = np.nan

## 5. Handling Outliers
Outliers are extreme values that can distort models. We detect them with the **IQR method**:

`lower = Q1 - 1.5 x IQR` and `upper = Q3 + 1.5 x IQR`, where `IQR = Q3 - Q1`.

In [ ]:
plt.figure(figsize=(16, 4))
for i, col in enumerate(numeric_cols, 1):
    plt.subplot(1, 5, i)
    sns.boxplot(y=df_clean[col])
    plt.title(col)
plt.tight_layout()
plt.show()

**Observation:**

*(Write what you notice after running the cell above.)*

### Detect outliers using IQR
Missing values (`NaN`) are ignored automatically.

In [ ]:
def iqr_bounds(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

rows = []
for col in numeric_cols:
    lower, upper = iqr_bounds(df_clean[col])
    n_out = ((df_clean[col] < lower) | (df_clean[col] > upper)).sum()
    rows.append({"feature": col, "lower": round(lower, 2), "upper": round(upper, 2), "n_outliers": n_out})

pd.DataFrame(rows)

**Observation:**

*(Write what you notice after running the cell above.)*

#### Important note on removing outliers
In medical data, an extreme value can be a **real and clinically important patient** (for example very high blood pressure), not an error.
Before removing anything ask: *is this a mistake, or a rare but valid case?*
Removing valid rare cases can remove exactly the patients the model should learn to recognise.

So instead of deleting rows, we demonstrate **capping** (percentile method): values below the 1st percentile or above the 99th percentile are replaced by those limits.

In [ ]:
lower_cap = df_clean["Cholesterol"].quantile(0.01)
upper_cap = df_clean["Cholesterol"].quantile(0.99)

chol_capped = df_clean["Cholesterol"].clip(lower_cap, upper_cap)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.boxplot(x=df_clean["Cholesterol"], ax=axes[0]); axes[0].set_title("Cholesterol (before capping)")
sns.boxplot(x=chol_capped, ax=axes[1]);             axes[1].set_title("Cholesterol (after capping)")
plt.tight_layout()
plt.show()

**Observation:**

*(Write what you notice after running the cell above.)*

**Decision for the project:** keep, cap or remove outliers? Write your choice and reason here.

## 6. Data Transformation - Scaling (demonstration)
Features are on very different scales (`Cholesterol` ~ hundreds, `Oldpeak` ~ 0-6). Models that use distances or gradients (KNN, SVM, Logistic Regression, Neural Networks, PCA) are affected; tree-based models (Decision Tree, Random Forest) are not.

For this demonstration we use the median-imputed copy.

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

demo = df_median[numeric_cols].copy()
demo.describe().loc[["mean", "std", "min", "max"]].round(2)

### Min-Max normalization
Rescales each feature to the range 0 to 1: `x' = (x - min) / (max - min)`.

In [ ]:
minmax = MinMaxScaler()
demo_minmax = pd.DataFrame(minmax.fit_transform(demo), columns=numeric_cols)
demo_minmax.describe().loc[["mean", "std", "min", "max"]].round(2)

**Observation:**

*(Write what you notice after running the cell above.)*

### Z-score standardization
Mean 0 and standard deviation 1: `z = (x - mean) / std`.

In [ ]:
standard = StandardScaler()
demo_standard = pd.DataFrame(standard.fit_transform(demo), columns=numeric_cols)
demo_standard.describe().loc[["mean", "std", "min", "max"]].round(2)

**Observation:**

*(Write what you notice after running the cell above.)*

> **Leakage reminder:** in this notebook `fit_transform` ran on the whole dataset only to show the effect.
> In the real pipeline, the scaler is **fit on the training set** and only **transform** is applied to the test set.

## 7. Check correlation before PCA
PCA is useful when features are strongly correlated (they contain overlapping information).
- Close to **1** or **-1**: strong linear relationship
- Close to **0**: weak or no linear relationship

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(demo_standard.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap (Before PCA)")
plt.show()

**Observation:**

*(Write what you notice after running the cell above.)*

## 8. Data Reduction - PCA (optional demonstration)
If the correlations above are weak, PCA will not reduce the dimensions much. Look at the explained variance to judge.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=len(numeric_cols))
components = pca.fit_transform(demo_standard)

print("Explained variance ratio:", pca.explained_variance_ratio_.round(3))
print("Cumulative:              ", pca.explained_variance_ratio_.cumsum().round(3))

In [ ]:
plt.figure(figsize=(6, 5))
sns.scatterplot(x=components[:, 0], y=components[:, 1],
                hue=df_median[target_col].values, palette="Set1", alpha=0.7)
plt.title("PCA Projection")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.show()

**Observation:**

*(Write what you notice after running the cell above.)*

**Decision:** is PCA worth using in the final model? Write your answer and reason.

## 9. Save the cleaned dataset
We save `df_clean`: duplicates removed and hidden zeros converted to `NaN`. Notebook 03 (feature engineering) reads this file.
We deliberately do **not** impute or scale here, so the model notebooks can do those steps correctly after the train/test split.

In [ ]:
os.makedirs("../data/processed", exist_ok=True)
df_clean.to_csv("../data/processed/heart_clean.csv", index=False)

print("Saved:", df_clean.shape)
print("Missing values kept as NaN:")
print(df_clean.isna().sum()[df_clean.isna().sum() > 0])

---
### Task
Repeat the IQR outlier detection for `Age` and `MaxHR` only, print how many outliers each has, and explain whether you would treat them as errors or valid values.